# Step 2 – Heating network expansion (DHNx)
**MJ2505 Project 2 – District heating network optimisation**

| | |
|---|---|
| **Goal** | Expand Step 1: **10 consumers** (Step 1's five + five new), **5 producers** (Step 1's two + three new, incl. a second **heat pump**), **5 pipe types** DN-25/30/40/50/60. |
| **Reads** | `results/step1/` (for the comparison) · demand profiles · `data/pipe_types.csv` · `data/producers_technology.csv` (Elisabeth) |
| **Writes** | `data/step2/` (DHNx input, scaled demand) · `results/step2/` – incl. **`network_losses.csv`, used by Step 4** |
| **Grading** | D – verify feasibility; *quantify* how topology, pipe selection, investment/operating cost and losses change vs. Step 1; identify underused producers / binding pipe capacities |
| **Tutorial reference** | `DHNx Tutorial/DHNx.ipynb` → *discrete_DN_numbers* |

Run `step1_base_network.ipynb` first.

## 0. Imports
(The environment is created by the setup cell at the top of Step 1.)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

import helpers as h      # section 0: assumptions, 4: Vasastan network + DHNx runs
from helpers import DATA, RESULTS, SOLVER

pd.set_option("display.precision", 2)
pd.set_option("display.width", 160)
OUT = RESULTS / "step2"
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# Diagnostic (from the course "Prerequisites" document): which Python / packages / solver are used?
import sys, shutil
import pandas as pd
import oemof.solph as solph
import dhnx
from helpers import SOLVER

print("Python executable:", sys.executable)
print("Python version:   ", sys.version.split()[0])
print("DHNx version:     ", dhnx.__version__)
print("oemof.solph:      ", solph.__version__)
print("pandas:           ", pd.__version__)
print("Solver used:      ", SOLVER, "->", shutil.which("cbc") or "HiGHS via the 'highspy' pip package")

## 1. What changes from Step 1

Everything from Step 1 is kept: the demand scale factor 0.4, simultaneity 1, Didrik's pipe table, the two added modelling rules (one pipe per edge, one connected network), and the Step 1 nodes. The network grows east to Upplandsgatan and Norrtullsgatan.

New producers (technology data in `data/producers_technology.csv`, from Elisabeth):

| Id | Technology | Site | Siting logic |
|---|---|---|---|
| P2 | Electric boiler | Next to heat pump P0 at Odenplan station | Shares P0's grid connection; peak and backup unit |
| P3 | Biogas boiler | Odengatan 69 (assumed boiler room) | Stockholm's inner-city gas grid carries biogas; peak and reserve |
| P4 | Heat pump 2 | Hemköp supermarket, Karlbergsvägen 24 | Recovers refrigeration heat (food retail waste heat) |

In [ ]:
inp = h.write_step_inputs(2)
demand = inp["demand"]
consumers = inp["consumers"][["key", "name", "building_type", "profile", "step"]].copy()
consumers["peak_kW"] = demand.max().values
consumers["annual_MWh"] = demand.sum().values / 1000
consumers["new_in_step2"] = consumers.step == 2
consumers.drop(columns="step")

In [ ]:
prods = inp["producers"][["key", "name", "technology", "step", "note"]]
tech = pd.read_csv(h.DATA / "producers_technology.csv")
prods.merge(tech[["id", "efficiency_or_COP", "capex_EUR_per_MW", "var_OM_EUR_per_MWh"]],
            left_on="key", right_on="id").drop(columns="id")

In [ ]:
sum_peaks = demand.max().sum(); coincident = demand.sum(axis=1).max()
print(f"Sum of individual peaks : {sum_peaks:6.1f} kW  (design load)")
print(f"Coincident peak         : {coincident:6.1f} kW   coincidence factor {coincident / sum_peaks:.2f}")
print(f"Annual demand           : {demand.sum().sum() / 1000:6.1f} MWh")
h.RESULTS.joinpath("step2").mkdir(parents=True, exist_ok=True)
ax = h.plot_dh_network(2, title="Step 2: candidate network (lengths in m)")
plt.gcf().savefig(h.RESULTS / "step2" / "map_candidates.png", dpi=150, bbox_inches="tight")

## 2. Optimisation

This is the same formulation as Step 1 (see section 4 there), now with five pipe types and five producers. The problem is larger, so it takes longer to solve: every new candidate edge adds 2 directions × 5 types = 10 binary variables.

In [ ]:
net = h.run_dhnx(2)
info = h.solver_info(net)
info

In [ ]:
pipes = h.pipe_results(net, 2)
built = pipes[pipes.built]
summary = h.summary(net, 2, pipes)
supply = h.producer_supply(net, 2)
print(f"Demand + losses = {summary['design demand (sum of peaks) [kW]'] + summary['heat losses at design hour [kW]']:.1f} kW,"
      f" producers supply {supply.supply_kW.sum():.1f} kW")
summary.round(1)

In [ ]:
ax = h.plot_dh_network(2, pipes, title="Step 2: optimised network (numbers = heat flow at design hour, kW)")
plt.gcf().savefig(h.RESULTS / "step2" / "map_result.png", dpi=150, bbox_inches="tight")

In [ ]:
cols = ["id", "street", "from_key", "to_key", "kind", "length", "hp_type", "capacity",
        "design_flow_kW", "losses", "utilisation", "costs"]
built[cols].rename(columns={"capacity": "capacity_kW", "losses": "loss_kW",
                            "costs": "cost_EUR"}).sort_values(["kind", "hp_type"])

In [ ]:
supply

## 3. Comparison with Step 1

In [ ]:
s1 = pd.read_csv(h.RESULTS / "step1" / "summary.csv", index_col=0)["value"]
s1_pipes = pd.read_csv(h.RESULTS / "step1" / "pipes_result.csv")
compare = pd.DataFrame({"Step 1": s1, "Step 2": summary})
compare["Step 2 / Step 1"] = compare["Step 2"] / compare["Step 1"]
extra = pd.DataFrame({
    "Step 1": [s1["pipe investment [EUR]"] / s1["design demand (sum of peaks) [kW]"],
               100 * s1["annual heat loss [MWh/yr]"] / s1["annual demand [MWh/yr]"]],
    "Step 2": [summary["pipe investment [EUR]"] / summary["design demand (sum of peaks) [kW]"],
               100 * summary["annual heat loss [MWh/yr]"] / summary["annual demand [MWh/yr]"]]},
    index=["investment per kW design load [EUR/kW]", "annual losses / annual demand [%]"])
extra["Step 2 / Step 1"] = extra["Step 2"] / extra["Step 1"]
pd.concat([compare, extra]).round(2)

In [ ]:
# Pipe selection on the Step 1 edges: did the same street get a different pipe?
key = ["from_key", "to_key"]
same = s1_pipes[s1_pipes.built][key + ["street", "hp_type", "design_flow_kW"]].merge(
    pipes[key + ["hp_type", "built", "design_flow_kW"]], on=key, how="left", suffixes=("_step1", "_step2"))
same["hp_type_step2"] = same.hp_type_step2.where(same.built, "not built")
same.drop(columns="built")

In [ ]:
mix = pd.DataFrame({
    "Step 1 length [m]": s1_pipes[s1_pipes.built].groupby("hp_type").length.sum(),
    "Step 2 length [m]": built.groupby("hp_type").length.sum(),
    "Step 2 pipes": built.groupby("hp_type").size(),
}).reindex(h.PIPE_TYPES_BY_STEP[2]).fillna(0)
mix["Step 2 used as"] = built.groupby("hp_type").kind.agg(
    lambda s: ", ".join(f"{k} x{n}" for k, n in s.value_counts().items()))
mix

In [ ]:
binding = built[built.utilisation > 0.95][["street", "from_key", "to_key", "kind", "hp_type",
                                             "capacity", "design_flow_kW", "losses", "utilisation"]]
binding

## 4. Answers to the Step 2 questions

Numbers refer to the run above.

**How did more consumers and producers change the topology and pipe selection?**
- **The network is not "Step 1 plus an extension".** The optimiser redesigned it:
  - The Step 1 trunk down Dalagatan (DN-50 from the heat pump P0) is no longer built.
  - P0 now feeds north-east along Vanadisvägen (DN-60) towards the new consumers, Matteusskolan and Upplandsgatan 82.
  - The new supermarket heat pump P4 sits in the middle of the network and takes over the centre: the school Vasa Real, the hotel and Västmannagatan.
  - The biomass boiler P1 still feeds Sabbatsberg (hospital and care home).
- **Why:** with producers at more locations, heat travels shorter distances, so most pipes carry less.
- **What this means for a real expansion:** the Step 1 pipes would already be in the ground. DHNx can handle that by marking pipes as `existing`, which is a possible extension and a limitation of this comparison.

**Did the optimiser choose different pipe types than in Step 1? Why?**
Yes. In Step 1 the only options were 53.6 kW (DN-30) or 208.9 kW (DN-50), so every load between those got an oversized DN-50. With five sizes, the optimiser picks the cheapest pipe that still fits each flow:
- **DN-40 (115 kW)** replaces DN-50 on the service pipes to the school, hotel and hospital (73–109 kW), and on mid-sized mains.
- **DN-25 (36 kW)** is used for the smallest building and for the links that only exist to keep the network connected.
- **DN-60 (339 kW)** appears only right next to the large producers (P0 and P4).

**Are some producers associated with much lower supplied capacity?**
Yes:
- **The electric boiler (P2)** delivers about 0 kW. It only covers the loss in its own short pipe.
- **The biogas boiler (P3)** delivers 53.5 kW, exactly the capacity of its DN-30 connection.
- **The biomass boiler (P1)** is limited to 208.9 kW by its DN-50 connection (100 % utilised).

The mechanism matters for the exam. **DHNx has no production costs**: all heat is free, so which producer supplies how much is decided only by pipe costs. P2 is co-located with P0, so anything it could deliver, P0 can deliver through the same junction without an extra pipe. P3 is 180 m away at the east edge, and a bigger connection pipe costs more than letting the central producers cover the load. These splits are a network artefact, not an economic dispatch. The economic split between producers is decided in Step 3 with oemof. The connection pipe sizes found here then become physical limits worth checking (see the open issues in the README).

**How do investment, operating costs and losses compare with Step 1?**
- **Scale:** design load ×2.3 (349 → 796 kW).
- **Investment:** ×1.7 (€0.43M → €0.75M). Per kW of design load it drops from about 1235 to 940 €/kW, an economy of scale from denser demand and right-sized pipes.
- **Losses at design hour:** ×1.8 (12.7 → 23.1 kW). As a share of annual demand they fall from about 11 % to 7 %.
- **Operating costs:** inside DHNx the only operating effect is the heat lost in the pipes, 111 MWh/yr in Step 1 and 202 MWh/yr in Step 2. Its cost depends on the heat production cost, which Step 4 adds by running oemof with the losses included.
- **Losses are constant over the year,** because `l_factor = 0`: the loss depends on whether a pipe exists, not on the flow. That is physically reasonable, since losses are driven by the temperature difference to the ground. It also means losses are a much larger share of demand in summer.

**Computational complexity:** Step 1 has about 200 variables (62 binary) and solves in about 1 s. Step 2 has about 860 variables (275 binary) and takes about 20 s. The problem is 4 times larger but about 20 times slower, which is typical for MILPs, where the number of combinations grows quickly with binary variables.

## 5. What DHNx does without the "one connected network" rule (comparison)

In [ ]:
import networkx as nx
net_free = h.run_dhnx(2, connected=False)
pipes_free = h.pipe_results(net_free, 2)
b_free = pipes_free[pipes_free.built]
islands = nx.number_connected_components(nx.from_pandas_edgelist(b_free, "from_key", "to_key"))
print(f"Without connectivity rule: {islands} separate islands, investment {b_free.costs.sum():,.0f} EUR")
print(f"With connectivity rule   : 1 network, investment {built.costs.sum():,.0f} EUR "
      f"(+{100 * (built.costs.sum() / b_free.costs.sum() - 1):.0f} %)")
print("Producer supply without the rule:", h.producer_supply(net_free, 2).set_index("producer").supply_kW.round(1).to_dict())
ax = h.plot_dh_network(2, pipes_free, title="Step 2 without the connectivity rule (comparison only)")

Without the rule, the electric boiler is not connected at all and the area splits into four islands. Connecting everything costs about 28 % more. That premium buys the ability to move heat between producer areas, which Steps 3–5 rely on. In Step 1 the same premium is under 2 %, because one short DN-30 link was enough.

## 6. Export for Step 3 and Step 4

Step 4 needs the network losses as a time series in the same units and resolution as the demand: hourly, in kW (= kWh per hour).

- `l_factor = 0`, so the DHNx loss is a constant kW value for every hour.
- `Q_total = Q_demand + Q_losses`, as in the assignment.
- Step 4 reads `results/step2/network_losses.csv` (`total_loss_kW`, `annual_loss_MWh`). `heat_demand_plus_losses_hourly.csv` is the same thing as a full hourly table, for checking.

In [ ]:
out = h.RESULTS / "step2"
loss_kW = built.losses.sum()
hourly = pd.DataFrame({"demand_kW": demand.sum(axis=1)})
hourly["losses_kW"] = loss_kW
hourly["total_kW"] = hourly.demand_kW + hourly.losses_kW
hourly.to_csv(out / "heat_demand_plus_losses_hourly.csv")

pipes.to_csv(out / "pipes_result.csv", index=False)
h.export_network_losses(pipes, out / "network_losses.csv")          # read by Step 4
built[["street", "from_key", "to_key", "hp_type", "length", "losses"]].to_csv(out / "losses_per_pipe.csv", index=False)
supply.to_csv(out / "producer_supply.csv", index=False)
summary.to_csv(out / "summary.csv", header=["value"])
info.to_csv(out / "solver_info.csv", header=["value"])
print(f"Losses: {loss_kW:.2f} kW every hour = {loss_kW * 8760 / 1000:.1f} MWh/yr "
      f"({100 * loss_kW * 8760 / hourly.demand_kW.sum():.1f} % of annual demand)")
print("Saved to", out.relative_to(h.ROOT), ":", sorted(p.name for p in out.iterdir()))
hourly.describe().round(1)